In [10]:
import pandas as pd

print("Part A - Understanding individual dataset")
cust = pd.read_csv('customers.csv')
order = pd.read_csv('orders.csv')
prod = pd.read_csv('products.csv')

print("\nDimension of Customer file:", cust.shape)
print("\nDimension of orders file:", order.shape)
print("\nDimension of products file:", prod.shape)

print("\nColumns in each dataset")
print(cust.columns)
print(order.columns)
print(prod.columns)

common_cols_cust_order = [col for col in cust.columns if col in order.columns]
common_cols_order_prod = [col for col in order.columns if col in prod.columns]
common_cols_all = [col for col in cust.columns if col in order.columns and col in prod.columns]

print("\nCommon columns between cust and order:", common_cols_cust_order)
print("Common columns between order and prod:", common_cols_order_prod)
print("Common columns across all three:", common_cols_all)

cust_order = pd.merge(order, cust, on="Customer_ID", how="inner")
order_prod = pd.merge(order, prod, on="Product_ID", how="inner")
full_relation = pd.merge(cust_order, prod, on="Product_ID", how="inner")

print("\ncust-order merged shape:", cust_order.shape)
print("order-prod merged shape:", order_prod.shape)
print("Full relationship merged shape:", full_relation.shape)

Part A - Understanding individual dataset

Dimension of Customer file: (3000, 5)

Dimension of orders file: (15000, 5)

Dimension of products file: (100, 4)

Columns in each dataset
Index(['Customer_ID', 'Customer_Name', 'City', 'Age', 'Gender'], dtype='object')
Index(['Order_ID', 'Customer_ID', 'Product_ID', 'Quantity', 'Order_Date'], dtype='object')
Index(['Product_ID', 'Product_Name', 'Category', 'Price'], dtype='object')

Common columns between cust and order: ['Customer_ID']
Common columns between order and prod: ['Product_ID']
Common columns across all three: []

cust-order merged shape: (15000, 9)
order-prod merged shape: (15000, 8)
Full relationship merged shape: (15000, 12)


In [18]:
print("Part B — Data Cleaning ")

print("Missing values in cust:\n", cust.isnull().sum())
print("Missing values in prod:\n", prod.isnull().sum())
print("Missing values in order:\n", order.isnull().sum())

print("\nDuplicate records in cust:", cust.duplicated().sum())
print("Duplicate records in prod:", prod.duplicated().sum())
print("Duplicate records in order:", order.duplicated().sum())

invalid_customer_ids = order[~order["Customer_ID"].isin(cust["Customer_ID"])]
invalid_product_ids = order[~order["Product_ID"].isin(prod["Product_ID"])]

print("\nInvalid Customer_IDs in order:\n", invalid_customer_ids)
print("Invalid Product_IDs in order:\n", invalid_product_ids)

if "Order_Date" in order.columns:
    order["Order_Date"] = pd.to_datetime(order["Order_Date"], errors="coerce")
    print("\nConverted Order_Date column type:", order["Order_Date"].dtype)

missing_customers = order[~order["Customer_ID"].isin(cust["Customer_ID"])]
print("\nOrders with invalid customers:\n", missing_customers)


missing_products = order[~order["Product_ID"].isin(prod["Product_ID"])]
print("\nOrders with invalid products:\n", missing_products)

Part B — Data Cleaning 
Missing values in cust:
 Customer_ID      0
Customer_Name    0
City             0
Age              0
Gender           0
dtype: int64
Missing values in prod:
 Product_ID      0
Product_Name    0
Category        0
Price           0
dtype: int64
Missing values in order:
 Order_ID       0
Customer_ID    0
Product_ID     0
Quantity       0
Order_Date     0
dtype: int64

Duplicate records in cust: 0
Duplicate records in prod: 0
Duplicate records in order: 0

Invalid Customer_IDs in order:
 Empty DataFrame
Columns: [Order_ID, Customer_ID, Product_ID, Quantity, Order_Date]
Index: []
Invalid Product_IDs in order:
 Empty DataFrame
Columns: [Order_ID, Customer_ID, Product_ID, Quantity, Order_Date]
Index: []

Converted Order_Date column type: datetime64[ns]

Orders with invalid customers:
 Empty DataFrame
Columns: [Order_ID, Customer_ID, Product_ID, Quantity, Order_Date]
Index: []

Orders with invalid products:
 Empty DataFrame
Columns: [Order_ID, Customer_ID, Product_ID, Q

In [22]:
print("Part C — Combining Data")

order_cust = pd.merge(order, cust, on="Customer_ID", how="inner")
print(order_cust)

order_cust_prod = pd.merge(order_cust, prod, on="Product_ID", how="inner")
print(order_cust_prod)

final_df = order_cust_prod[[
    "Order_ID",
    "Customer_ID",
    "Customer_Name",
    "City",
    "Age",
    "Gender",
    "Product_ID",
    "Product_Name",
    "Category",
    "Price",
    "Quantity",
    "Order_Date"
]]

print(final_df.head())

Part C — Combining Data
      Order_ID Customer_ID Product_ID  Quantity          Order_Date  \
0      O000001      C00861      P0027         4 2025-01-01 00:00:00   
1      O000002      C01295      P0047         3 2025-01-01 02:00:00   
2      O000003      C01131      P0099         4 2025-01-01 04:00:00   
3      O000004      C01096      P0006         1 2025-01-01 06:00:00   
4      O000005      C01639      P0060         2 2025-01-01 08:00:00   
...        ...         ...        ...       ...                 ...   
14995  O014996      C01275      P0078         3 2028-06-03 14:00:00   
14996  O014997      C00946      P0017         5 2028-06-03 16:00:00   
14997  O014998      C02228      P0022         5 2028-06-03 18:00:00   
14998  O014999      C00293      P0030         2 2028-06-03 20:00:00   
14999  O015000      C00584      P0081         1 2028-06-03 22:00:00   

       Customer_Name        City  Age  Gender  
0       Customer_861     Kolkata   51  Female  
1      Customer_1295   Bang

In [31]:
print("Part D — Business Calculations")

final_df.loc[:, "Order_Value"] = final_df["Price"] * final_df["Quantity"]

total_revenue = final_df["Order_Value"].sum()
print("\nTotal Revenue:", total_revenue)


avg_order_value = final_df["Order_Value"].mean()
print("Average Order Value:", avg_order_value)


revenue_by_category = final_df.groupby("Category")["Order_Value"].sum()
print("\nRevenue by Product Category:\n", revenue_by_category)


revenue_by_city = final_df.groupby("City")["Order_Value"].sum()
print("\nRevenue by City:\n", revenue_by_city)


top_customers = final_df.groupby(["Customer_ID","Customer_Name"])["Order_Value"].sum().nlargest(10)
print("\nTop 10 Customers by Spending:\n", top_customers)


top_products = final_df.groupby(["Product_ID","Product_Name"])["Order_Value"].sum().nlargest(10)
print("\nTop 10 Products by Revenue:\n", top_products)


most_freq_product = final_df["Product_Name"].mode()[0]
print("\nMost Frequently Purchased Product:", most_freq_product)


city_customer_count = final_df.groupby("City")["Customer_ID"].nunique()
top_city_customers = city_customer_count.idxmax()
print("\nCity with Highest Number of Customers:", top_city_customers)


top_city_revenue = revenue_by_city.idxmax()
print("\nCity Generating Highest Revenue:", top_city_revenue)



Part D — Business Calculations

Total Revenue: 2233171871.3
Average Order Value: 148878.12475333334

Revenue by Product Category:
 Category
Beauty             2.966918e+08
Books              3.050563e+08
Clothing           4.261489e+08
Electronics        3.312926e+08
Home Appliances    3.313110e+08
Sports             5.426713e+08
Name: Order_Value, dtype: float64

Revenue by City:
 City
Bangalore     2.637059e+08
Chennai       2.519685e+08
Coimbatore    2.831225e+08
Delhi         2.654541e+08
Hyderabad     2.932639e+08
Kolkata       3.155500e+08
Mumbai        2.803566e+08
Pune          2.797505e+08
Name: Order_Value, dtype: float64

Top 10 Customers by Spending:
 Customer_ID  Customer_Name
C00019       Customer_19      3176078.87
C02897       Customer_2897    2528868.53
C01915       Customer_1915    2476932.74
C02709       Customer_2709    2400905.35
C00187       Customer_187     2394078.09
C00685       Customer_685     2352464.63
C01297       Customer_1297    2345815.39
C00259       C

In [33]:
print("Part E — Customer Analysis")


final_df = final_df.copy()

bins = [18, 30, 45, 60, 120]
labels = ["18–30", "31–45", "46–60", "61+"]
final_df["Age_Group"] = pd.cut(final_df["Age"], bins=bins, labels=labels, right=True)


revenue_by_age_group = final_df.groupby("Age_Group", observed=False)["Order_Value"].sum()
print("\nRevenue by Age Group:\n", revenue_by_age_group)


spending_by_gender = final_df.groupby("Gender")["Order_Value"].sum()
print("\nSpending by Gender:\n", spending_by_gender)


customer_spending = final_df.groupby(["Customer_ID","Customer_Name"])["Order_Value"].sum()
threshold = customer_spending.quantile(0.95)
high_value_customers = customer_spending[customer_spending >= threshold]
print("\nHigh-Value Customers:\n", high_value_customers)

orders_per_customer = final_df.groupby("Customer_ID")["Order_ID"].nunique()
multi_order_customers = (orders_per_customer > 1).sum()
percentage_multi_order = (multi_order_customers / orders_per_customer.shape[0]) * 100
print("\nPercentage of Customers with >1 Order:", percentage_multi_order)


Part E — Customer Analysis

Revenue by Age Group:
 Age_Group
18–30    5.208381e+08
31–45    6.223524e+08
46–60    6.599237e+08
61+      3.922680e+08
Name: Order_Value, dtype: float64

Spending by Gender:
 Gender
Female    1.078972e+09
Male      1.154200e+09
Name: Order_Value, dtype: float64

High-Value Customers:
 Customer_ID  Customer_Name
C00012       Customer_12      1690816.86
C00019       Customer_19      3176078.87
C00029       Customer_29      1593831.14
C00072       Customer_72      1626112.61
C00139       Customer_139     1670641.54
                                 ...    
C02897       Customer_2897    2528868.53
C02929       Customer_2929    1617063.54
C02972       Customer_2972    1654429.34
C02974       Customer_2974    1787219.20
C02988       Customer_2988    2105275.02
Name: Order_Value, Length: 149, dtype: float64

Percentage of Customers with >1 Order: 96.80779569892472


In [34]:
print("Part F — Advanced Analysis")

customer_spending = final_df.groupby(["Customer_ID","Customer_Name"])["Order_Value"].sum().sort_values(ascending=False)
print("\nCustomer Spending Ranking:\n", customer_spending)


customer_spending = final_df.groupby(["Customer_ID","Customer_Name"])["Order_Value"].sum()
customer_spending_sorted = customer_spending.sort_values(ascending=False)
top_10_percent_count = int(len(customer_spending_sorted) * 0.10)
top_10_percent_customers = customer_spending_sorted.head(top_10_percent_count)
print("\nTop 10% Customers by Spending:\n", top_10_percent_customers)



category_by_city = final_df.groupby(["City","Category"])["Order_Value"].sum()
most_profitable_category_city = category_by_city.groupby(level=0).idxmax()
print("\nMost Profitable Product Category for Each City:\n", most_profitable_category_city)


popular_product_category = final_df.groupby(["Category","Product_Name"])["Quantity"].sum()
most_popular_product_category = popular_product_category.groupby(level=0).idxmax()
print("\nMost Popular Product in Each Category:\n", most_popular_product_category)


final_df["Order_Date"] = pd.to_datetime(final_df["Order_Date"], errors="coerce")
monthly_revenue = final_df.groupby(final_df["Order_Date"].dt.to_period("M"))["Order_Value"].sum()
print("\nMonthly Revenue:\n", monthly_revenue)


highest_revenue_month = monthly_revenue.idxmax()
print("\nMonth with Highest Revenue:", highest_revenue_month)


Part F — Advanced Analysis

Customer Spending Ranking:
 Customer_ID  Customer_Name
C00019       Customer_19      3176078.87
C02897       Customer_2897    2528868.53
C01915       Customer_1915    2476932.74
C02709       Customer_2709    2400905.35
C00187       Customer_187     2394078.09
                                 ...    
C01172       Customer_1172       7497.06
C00879       Customer_879        4561.62
C00577       Customer_577        2606.64
C02035       Customer_2035       2383.56
C01558       Customer_1558       1954.98
Name: Order_Value, Length: 2976, dtype: float64

Top 10% Customers by Spending:
 Customer_ID  Customer_Name
C00019       Customer_19      3176078.87
C02897       Customer_2897    2528868.53
C01915       Customer_1915    2476932.74
C02709       Customer_2709    2400905.35
C00187       Customer_187     2394078.09
                                 ...    
C01044       Customer_1044    1344617.29
C00465       Customer_465     1344033.24
C00363       Customer_363     